# Candidate Generation: Static Item-Item Collaborative Filtering

Builds a binary user-item matrix from the training split, derives item-item
Top-K cosine neighbours, and writes target-split Top-1500 candidates.

The matrix uses only training data: latest `full_history_rows` per user plus all
`candidates_train` clicks. Candidates are scored causally, history articles are
excluded, and eligibility is `available_from <= timestamp`.


## 1. Setup

In [ ]:
from pathlib import Path
import heapq
import sys

import numpy as np
import polars as pl
from scipy import sparse

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_all.py").is_file()
)
sys.path.insert(0, str(ROOT / "5_Models" / "5_01_Candidate_Generation"))

from cf_models import build_interaction_matrix, load_context, resolve_device
from cg_result_writer import CandidateWriter

ITEM_K = 100
M_VALUES = (50, 100, 200, 500)
CANDIDATE_CAP = max(M_VALUES)
TARGET_SPLIT = "validation"
REBUILD = True
CF_DEVICE = "auto"
FEATURES = ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"
RESULTS = ROOT / "results"

MODEL_NAME = "cf_item_item"
ARTIFACT_DIR = (
    ROOT / "5_Models" / "5_01_Candidate_Generation" / "artifacts" / MODEL_NAME
)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)


## 2. Data ingestion and user-item matrix

In [2]:
context = load_context(ROOT)
matrix, matrix_manifest = build_interaction_matrix(ROOT, context, rebuild=REBUILD)

print(
    f"Matrix: {matrix.shape[0]:,} users x {matrix.shape[1]:,} articles, "
    f"{matrix.nnz:,} nonzeros"
)
matrix_manifest


Matrix: 50,001 users x 51,283 articles, 936,005 nonzeros


{'version': 'cf-static-v1',
 'created_at': '2026-11-03T10:21:12.126423+00:00',
 'cutoff': '2019-11-12 23:59:25',
 'rows': 50001,
 'columns': 51283,
 'nonzero': 936005,
 'values': 'binary',
 'source': 'latest request_profiles_train history plus candidates_train clicks'}

## 3. Item-item similarities

In [3]:
def _normalised_columns(matrix):
    counts = np.asarray(matrix.sum(axis=0)).ravel().astype(np.float32)
    inverse = np.zeros_like(counts)
    nonzero = counts > 0
    inverse[nonzero] = 1.0 / np.sqrt(counts[nonzero])
    return matrix.T.tocsr().multiply(inverse[:, None]).tocsr(), counts


def _to_torch_csr(matrix, device):
    import torch

    return torch.sparse_csr_tensor(
        torch.from_numpy(matrix.indptr.astype(np.int64)).to(device),
        torch.from_numpy(matrix.indices.astype(np.int64)).to(device),
        torch.from_numpy(matrix.data.astype(np.float32)).to(device),
        size=matrix.shape,
        device=device,
        check_invariants=False,
    )


def _top_k_block(matrix, k, block):
    rows = matrix.shape[0]
    row_parts, col_parts, value_parts = [], [], []
    for start in range(0, rows, block):
        stop = min(start + block, rows)
        scores = (matrix[start:stop] @ matrix.T).tocsr()
        scores.sum_duplicates()
        scores.eliminate_zeros()
        for local in range(scores.shape[0]):
            lo, hi = scores.indptr[local], scores.indptr[local + 1]
            cols = scores.indices[lo:hi]
            values = scores.data[lo:hi]
            keep = cols != (start + local)
            cols = cols[keep]
            values = values[keep]
            if cols.size == 0:
                continue
            if cols.size > k:
                chosen = np.argpartition(values, -k)[-k:]
                cols = cols[chosen]
                values = values[chosen]
            order = np.argsort(-values, kind="stable")
            row_parts.append(np.full(cols.size, start + local, dtype=np.int32))
            col_parts.append(cols[order].astype(np.int32, copy=False))
            value_parts.append(values[order].astype(np.float32, copy=False))
    if not row_parts:
        return sparse.csr_matrix((rows, rows), dtype=np.float32)
    return sparse.csr_matrix(
        (
            np.concatenate(value_parts),
            (np.concatenate(row_parts), np.concatenate(col_parts)),
        ),
        shape=(rows, rows),
    )


def _top_k_block_gpu(matrix, k, block, device):
    import torch

    rows = matrix.shape[0]
    transpose = _to_torch_csr(matrix.T.tocsr(), device)
    row_parts, col_parts, value_parts = [], [], []
    with torch.no_grad():
        for start in range(0, rows, block):
            stop = min(start + block, rows)
            scores = torch.sparse.mm(
                _to_torch_csr(matrix[start:stop], device), transpose
            ).to_dense()
            local = torch.arange(stop - start, device=device)
            scores[local, torch.arange(start, stop, device=device)] = -1.0
            values, indices = torch.topk(scores, k, dim=1)
            values = values.cpu().numpy()
            indices = indices.cpu().numpy()
            keep = values > 0
            if not keep.any():
                continue
            row_index = np.broadcast_to(
                np.arange(start, stop, dtype=np.int32)[:, None], keep.shape
            )[keep]
            row_parts.append(row_index)
            col_parts.append(indices[keep].astype(np.int32, copy=False))
            value_parts.append(values[keep].astype(np.float32, copy=False))

    del transpose
    if not row_parts:
        return sparse.csr_matrix((rows, rows), dtype=np.float32)
    return sparse.csr_matrix(
        (
            np.concatenate(value_parts),
            (np.concatenate(row_parts), np.concatenate(col_parts)),
        ),
        shape=(rows, rows),
    )


def build_item_neighbors(matrix, k=ITEM_K, block=256, device="auto"):
    normalised, popularity = _normalised_columns(matrix)
    resolved = resolve_device(device)
    if resolved == "cuda":
        cosine = _top_k_block_gpu(normalised, k, block, resolved)
    else:
        cosine = _top_k_block(normalised, k, block)
    rows = np.repeat(
        np.arange(cosine.shape[0], dtype=np.int32), np.diff(cosine.indptr)
    )
    counts = cosine.copy()
    counts.data = counts.data * np.sqrt(
        popularity[rows] * popularity[cosine.indices]
    )
    return cosine, counts, popularity


## 4. Build and cache item-item neighbours

In [4]:
def load_or_build_item_neighbors(matrix, k=ITEM_K, *, rebuild=False, device="auto"):
    cosine_path = ARTIFACT_DIR / f"item_item_cosine_topk_{k}.npz"
    if not rebuild and cosine_path.exists():
        return sparse.load_npz(cosine_path)

    resolved = resolve_device(device)
    cosine, _, _ = build_item_neighbors(matrix, k=k, device=resolved)
    sparse.save_npz(cosine_path, cosine, compressed=True)
    return cosine


In [5]:
item_similarity = load_or_build_item_neighbors(
    matrix, k=ITEM_K, rebuild=REBUILD, device=CF_DEVICE
)

print(f"Item-item edges: {item_similarity.nnz:,}")
print(f"Items with neighbours: {(np.diff(item_similarity.indptr) > 0).sum():,}")


C:\Users\Yong Rui\AppData\Local\Temp\ipykernel_12008\2621839106.py:12: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at B:\src\torch\aten\src\ATen\Context.cpp:767.)
  return torch.sparse_csr_tensor(
C:\Users\Yong Rui\AppData\Local\Temp\ipykernel_12008\2621839106.py:12: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at B:\src\torch\aten\src\ATen\SparseCsrTensorImpl.cpp:51.)
  return torch.sparse_csr_tensor(


Item-item edges: 2,765,577
Items with neighbours: 35,976


## 5. Generate target-split candidates

In [6]:
class PopularityFallback:
    def __init__(self, available_from, popularity, news_code, limit=1500):
        self.popularity = popularity
        self.news_code = news_code
        self.limit = int(limit)
        order = np.argsort(available_from[1:], kind="stable") + 1
        self.order = order.astype(np.int64)
        self.order_available = available_from[self.order]
        self.position = 0
        self.heap = []

    def advance(self, timestamp):
        while (
            self.position < self.order.size
            and self.order_available[self.position] <= timestamp
        ):
            row = int(self.order[self.position])
            key = (float(self.popularity[row]), -int(self.news_code[row]), row)
            if len(self.heap) < self.limit:
                heapq.heappush(self.heap, key)
            elif key > self.heap[0]:
                heapq.heapreplace(self.heap, key)
            self.position += 1
        items = sorted(self.heap, key=lambda x: (-x[0], -x[1]))
        return np.asarray([item[2] for item in items], dtype=np.int32)


def _row_aggregate(matrix):
    matrix = matrix.tocsr()
    matrix.sum_duplicates()
    matrix.eliminate_zeros()
    return (
        matrix.indices.astype(np.int32, copy=False),
        matrix.data.astype(np.float32, copy=False),
    )


def select_top_k(
    candidate_rows,
    candidate_scores,
    history,
    popularity,
    news_code,
    available_from,
    timestamp,
    fallback,
    k=CANDIDATE_CAP,
):
    candidate_rows = np.asarray(candidate_rows, dtype=np.int64)
    candidate_scores = np.asarray(candidate_scores, dtype=np.float32)
    history = np.asarray(history, dtype=np.int64)
    selected = np.empty(0, dtype=np.int64)

    if candidate_rows.size:
        keep = available_from[candidate_rows] <= timestamp
        if history.size:
            keep &= ~np.isin(candidate_rows, history)
        candidate_rows = candidate_rows[keep]
        candidate_scores = candidate_scores[keep]
        if candidate_rows.size:
            order = np.lexsort(
                (
                    news_code[candidate_rows],
                    -popularity[candidate_rows],
                    -candidate_scores,
                )
            )
            selected = candidate_rows[order[:k]]

    if selected.size < k:
        remaining = np.asarray(fallback, dtype=np.int64)
        if history.size:
            remaining = remaining[~np.isin(remaining, history)]
        if selected.size:
            remaining = remaining[~np.isin(remaining, selected)]
        selected = np.concatenate([selected, remaining[: k - selected.size]])

    return selected.astype(np.int32, copy=False)


def _select_for_request(context, item_similarity, history, fallback, timestamp):
    history = np.unique(np.asarray(history, dtype=np.int64))
    if history.size:
        candidate_rows, candidate_scores = _row_aggregate(
            item_similarity[history]
        )
    else:
        candidate_rows = np.empty(0, dtype=np.int32)
        candidate_scores = np.empty(0, dtype=np.float32)
    return select_top_k(
        candidate_rows,
        candidate_scores,
        history,
        context.popularity,
        context.news_code,
        context.available_from,
        timestamp,
        fallback,
    )


In [7]:
def _load_profiles(split):
    parts = sorted((FEATURES / f"request_profiles_{split}").glob("*.parquet"))
    return pl.read_parquet(
        parts, columns=["impression_uid", "timestamp", "full_history_rows"]
    ).sort("timestamp", maintain_order=True)


def generate_recommendations(context, item_similarity, *, artifact_dir, rebuild=False):
    path = (
        Path(artifact_dir)
        / f"recommendations_{TARGET_SPLIT}_top{CANDIDATE_CAP}.parquet"
    )
    if path.exists() and not rebuild:
        return path

    profiles = _load_profiles(TARGET_SPLIT)
    uids = profiles["impression_uid"].to_numpy()
    timestamps = profiles["timestamp"].to_numpy().astype("datetime64[ns]")
    histories = profiles["full_history_rows"].to_list()
    starts = np.flatnonzero(np.r_[True, timestamps[1:] != timestamps[:-1]])
    ends = np.r_[starts[1:], len(timestamps)]

    fallback = PopularityFallback(
        context.available_from,
        context.popularity,
        context.news_code,
        limit=CANDIDATE_CAP + 1200,
    )
    cutoffs = np.arange(1, CANDIDATE_CAP + 1, dtype=np.int16)
    writer = CandidateWriter(path)
    try:
        for start, end in zip(starts, ends):
            timestamp = timestamps[start]
            fallback_rows = fallback.advance(timestamp)
            selected = np.vstack(
                [
                    _select_for_request(
                        context,
                        item_similarity,
                        histories[row],
                        fallback_rows,
                        timestamp,
                    )
                    for row in range(start, end)
                ]
            )
            writer.add(
                np.repeat(uids[start:end], CANDIDATE_CAP),
                np.tile(cutoffs, end - start),
                context.news_code[selected.reshape(-1)],
            )
    finally:
        writer.close()
    return path


In [8]:
item_path = generate_recommendations(
    context, item_similarity, artifact_dir=ARTIFACT_DIR, rebuild=REBUILD
)
print("Item-item rows:", pl.scan_parquet(item_path).select(pl.len()).collect().item())
print("Item-item file:", item_path)


Item-item rows: 15812000
Item-item file: c:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\artifacts\cf_item_item\recommendations_validation_top500.parquet


## 6. Evaluate Recall@M

In [9]:
from cg_evaluation import CandidateEvaluator

evaluator = CandidateEvaluator(root=ROOT, split=TARGET_SPLIT)
results = evaluator.evaluate_to_csv(
    item_path,
    model_name=MODEL_NAME,
    m_values=M_VALUES,
    output_dir=RESULTS,
)
results.sort(["m", "model"])


ValueError: invalid candidate cutoffs for 30674 impressions